# Season track map — one basin, one season

Plot **all storm tracks** for the 2005 North Atlantic season as `LineString`s, coloured by each storm's maximum Saffir-Simpson category, and annotate the strongest storm.

Uses `geometry='track'`, which returns one `LineString` per storm with summary attributes (`max_vmax_kt`, `min_mslp_hpa`, `max_category`, `ace`). Needs `pip install earthlens[tropycal]`.

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('tropycal_output')
OUT_DIR.mkdir(exist_ok=True)

tracks = None
try:
    tracks = EarthLens(
        variables=['north_atlantic'],
        data_source='tropycal',
        start='2005-06-01',
        end='2005-12-01',
        lat_lim=[0.0, 60.0],
        lon_lim=[-110.0, -10.0],
        source='hurdat',
        geometry='track',
        path=str(OUT_DIR),
    ).download(progress_bar=False)
    print(len(tracks), 'storm tracks')
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## Strongest storm of the season

In [ ]:
if tracks is not None and len(tracks):
    strongest = tracks.sort_values('max_vmax_kt', ascending=False).iloc[0]
    print(strongest['name'], '- max wind', strongest['max_vmax_kt'], 'kt,'
          ' min pressure', strongest['min_mslp_hpa'], 'hPa')

## Map all tracks coloured by max category

In [ ]:
if tracks is not None and len(tracks):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 6))
    tracks.plot(ax=ax, column='max_category', cmap='YlOrRd', legend=True,
                linewidth=1.5)
    strongest = tracks.sort_values('max_vmax_kt', ascending=False).iloc[0]
    cx, cy = strongest.geometry.centroid.x, strongest.geometry.centroid.y
    ax.annotate(strongest['name'], xy=(cx, cy), fontsize=11, fontweight='bold')
    ax.set_title('North Atlantic storm tracks, 2005 (coloured by max category)')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()